In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader, MatchedSiphraAcquisition, match_events
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used

# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio

colors = [
    ROOT.kRed,
    ROOT.kBlue,
    ROOT.kGreen + 2,
    ROOT.kOrange + 7,
    ROOT.kMagenta,
    ROOT.kCyan + 2,
    ROOT.kViolet,
    ROOT.kAzure + 7,
    ROOT.kPink + 7,
    ROOT.kTeal + 3,
    ROOT.kSpring + 5,
    ROOT.kYellow + 2,
    ROOT.kOrange + 1,
    ROOT.kBlue - 7,
    ROOT.kRed - 7,
    ROOT.kGray + 2,
    ROOT.kBlack,
    ROOT.kSpring - 7
]

In [2]:
#siphras = ['A', 'B']
siphras = ['A']

acqs = {}

files_A = sorted((project_root / 'data/261015').glob('0[1-2]*.csv'))

#files_A = sorted((project_root / 'data/261012').glob('A_0[5-8]*.csv'))
files_B = sorted((project_root / 'data/261012').glob('B_0[5-8]*.csv'))


acqs_A = [SiphraAcquisition(file) for file in files_A]
acqs_B = [SiphraAcquisition(file) for file in files_B]

acqs['A'] = acqs_A
acqs['B'] = acqs_B


acqs_AB = [acqs_A, acqs_B]

print(acqs_AB)
print(acqs)

[[SIPHRAAcquisition(File: '01_16Chtrigger_BG'), SIPHRAAcquisition(File: '02_SumChtrigger_BG')], [SIPHRAAcquisition(File: 'B_05_SumChTrigger_Cs137'), SIPHRAAcquisition(File: 'B_06_SumChTrigger_BG'), SIPHRAAcquisition(File: 'B_07_16ChTrigger_BG'), SIPHRAAcquisition(File: 'B_08_16ChTrigger_Cs137')]]
{'A': [SIPHRAAcquisition(File: '01_16Chtrigger_BG'), SIPHRAAcquisition(File: '02_SumChtrigger_BG')], 'B': [SIPHRAAcquisition(File: 'B_05_SumChTrigger_Cs137'), SIPHRAAcquisition(File: 'B_06_SumChTrigger_BG'), SIPHRAAcquisition(File: 'B_07_16ChTrigger_BG'), SIPHRAAcquisition(File: 'B_08_16ChTrigger_Cs137')]}


In [4]:

hists = {}

for siphra in siphras:
    print(siphra)
    acq = acqs[siphra]
    #print(acqs)
    
    triggers = {}
    sum_ch = [acq[1]]
    all_ch = [acq[0]]
    #sum_ch = [acq[1], acq[0]]
    #all_ch = [acq[2], acq[3]]
    #print(sum_ch)
    #print(all_ch)
    triggers["sum_ch"] = sum_ch
    triggers["all_ch"] = all_ch

    for trigger in (triggers):
        print(trigger)
        bg = triggers[trigger][0]
        sgnl = triggers[trigger][1]
        print(bg)
        print(sgnl)
        hist_bg = ROOT.TH1F(f"{trigger}_trigger_SIPHRA_{siphra}_BG", "", BITS12, 0, BITS12)
        hist_sgnl = ROOT.TH1F(f"{trigger}_trigger_SIPHRA_{siphra}_signal", "", BITS12, 0, BITS12)
        hist_sgnl.Fill(sgnl['Siphra_sum'])
        hist_bg.Fill(bg['Siphra_sum'])

        hist_bg.Scale(sgnl.exposure/bg.exposure)
    
        hist_clean = hist_sgnl.Clone(f"{trigger}_trigger_SIPHRA_{siphra}_clean")
        hist_clean.Add(hist_bg, -1)

        for hist in [hist_sgnl, hist_bg, hist_clean]:
            # hist.SetTitle(r"^{22}Na spectra - CI gain = 1/0.75 pC")
            hist.GetXaxis().SetTitle("ADC channel number")
            hist.GetYaxis().SetTitle(r"Normalized counts")

        hists[f"{trigger}_trigger_SIPHRA_{siphra}_signal"] = hist_sgnl
        hists[f"{trigger}_trigger_SIPHRA_{siphra}_clean"] = hist_clean
        hists[f"{trigger}_trigger_SIPHRA_{siphra}_BG"] = hist_bg
        del hist_sgnl, hist_clean, hist_bg
        
        

A
sum_ch


IndexError: list index out of range

In [ ]:
#channels = ["Siphra_sum", "Manual_sum"]


if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 2100, 1500)
cv.Divide(2, 2)

lgds = [ROOT.TLegend(0.48, 0.61, 0.75, 0.83) for _ in range(4)]

for n, siphra in enumerate(siphras):
    for i, trigger in enumerate(triggers):
        cv.cd(n*2+i+1)
        cv.cd(n*2+i+1).SetLogy()
        lg = lgds[n*2+i]

            
        hist_clean = hists[f"{trigger}_trigger_SIPHRA_{siphra}_clean"]
        hist_sgnl = hists[f"{trigger}_trigger_SIPHRA_{siphra}_signal"]
        hist_bg = hists[f"{trigger}_trigger_SIPHRA_{siphra}_BG"]
        
        lg.AddEntry(hist_bg, 'Background')
        lg.AddEntry(hist_sgnl, 'Source')
        lg.AddEntry(hist_clean, 'Background subtracted')
        hist_bg.SetLineColor(colors[1])
        hist_sgnl.SetLineColor(colors[2])
        hist_clean.SetLineColor(colors[3])
        hist_bg.SetStats(0)
        hist_bg.SetTitle(f"{trigger} trigger Cs-137 SIPHRA {siphra}")
        hist_sgnl.SetStats(0)
        hist_sgnl.SetTitle(f"{trigger} trigger Cs-137 SIPHRA {siphra}")
        hist_clean.SetStats(0)
        hist_clean.SetTitle(f"{trigger} trigger Cs-137 SIPHRA {siphra}")
        

        hist_sgnl.Draw("hist")
        hist_bg.Draw("sames hist")
        hist_clean.Draw("sames hist")
        
        ROOT.gPad.Update()
            
            
        lg.Draw()
        cv.Draw()

    


#print(hists)